# DX702 Coding Quiz: Week 12

## Imports

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm # For linear regression
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

from scipy.stats import t,skew
from tqdm import tqdm
from scipy.stats import skew
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LinearRegression


In [2]:
plt.rcParams['axes.titlesize']  = 10
plt.rcParams['axes.labelsize']  = 8
plt.rcParams['lines.linewidth'] = 0.5
plt.rcParams['lines.markersize'] = 3
plt.rcParams['axes.edgecolor']  = 'gray'
plt.rcParams['xtick.color']     = 'gray'
plt.rcParams['ytick.color'] = 'gray'
plt.rcParams['xtick.color'] = 'gray'
plt.rcParams['ytick.color'] = 'gray'
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['xtick.labelsize'] = 8

## Functions

## Question 1  
**10 Points**  

Use instrumental variables to find the effect of X on Z in homework_13.1.csv. Which of these is the closest? 

Option A.)

0.5861

Option B.)

0.3405

Option C.)

0.8723

Option D.)

0.2443



In [3]:
data = pd.read_csv('https://raw.githubusercontent.com/joshua-vonkorff/2025-summer-mod-6/refs/heads/main/homework_13.1.csv')
data = data.drop(columns=['Unnamed: 0'])
data

,Z,X,Y
0,1,0,0.056584
1,1,1,1.387519
2,0,1,4.289658
3,0,1,0.743003
4,0,1,2.718527
...,...,...,...
9995,1,1,-2.450135
9996,1,1,1.562418
9997,0,1,2.052854
9998,1,1,0.320198


In [4]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.sandbox.regression.gmm import IV2SLS

# Load the dataset
df = pd.read_csv('homework_13.1.csv')

# Display basic information about the dataset
print("Dataset shape:", df.shape)
print("\nColumn names:", df.columns.tolist())
print("\nFirst few rows:")
print(df.head())

print("\nBasic statistics:")
print(df.describe())

# Check for missing values
print("\nMissing values:")
print(df.isnull().sum())

# Now let's understand the instrumental variables setup
print("\n" + "="*50)
print("INSTRUMENTAL VARIABLES EXPLANATION")
print("="*50)

print("""
The question asks: "Use instrumental variables to find the effect of X on Z"

This means we want to estimate the causal effect of X on Z, but there might be 
endogeneity problems (like omitted variable bias, measurement error, or simultaneity).

In your dataset, you have:
- Z: The outcome variable (dependent variable)
- X: The treatment/explanatory variable whose effect we want to measure
- Y: This is likely the instrumental variable

For Y to be a valid instrument for X, it must satisfy two conditions:

1. RELEVANCE: Y must be correlated with X (the endogenous regressor)
   - We can test this with: Corr(Y, X) ≠ 0
   - Typically tested with F-statistic in first stage regression

2. EXCLUSION RESTRICTION: Y must affect Z only through its effect on X
   - Y should not have a direct effect on Z
   - This cannot be directly tested and must be justified theoretically

The two-stage least squares (2SLS) procedure:
Stage 1: Regress X on Y (and other exogenous variables)
         X = α₀ + α₁Y + ε₁
         Get predicted values: X̂

Stage 2: Regress Z on X̂ (the predicted values from stage 1)
         Z = β₀ + β₁X̂ + ε₂
         
The coefficient β₁ gives us the causal effect of X on Z.
""")

# Let's check the correlation between variables
print("\nCorrelation matrix:")
print(df.corr())

# First stage regression: X on Y
X = df['X']
Y = df['Y']
Z = df['Z']

# Add constant for regression
Y_with_const = sm.add_constant(Y)
first_stage = sm.OLS(X, Y_with_const).fit()

print(f"\nFIRST STAGE REGRESSION (X on Y):")
print(f"F-statistic: {first_stage.fvalue:.3f}")
print(f"R-squared: {first_stage.rsquared:.3f}")
print("Coefficient on Y:", first_stage.params.iloc[1])  # Use iloc for position-based indexing
print("P-value for Y coefficient:", first_stage.pvalues.iloc[1])

# Check instrument relevance (F-stat should be > 10 as rule of thumb)
if first_stage.fvalue > 10:
    print("✓ Instrument appears to be relevant (F > 10)")
else:
    print("⚠ Weak instrument concern (F < 10)")

# Second stage: Z on predicted X
X_fitted = first_stage.fittedvalues
X_fitted_with_const = sm.add_constant(X_fitted)
second_stage = sm.OLS(Z, X_fitted_with_const).fit()

print(f"\nSECOND STAGE REGRESSION (Z on fitted X):")
print("IV Estimate of X's effect on Z:", second_stage.params.iloc[1])  # Use iloc
print("Standard error:", second_stage.bse.iloc[1])
print("P-value:", second_stage.pvalues.iloc[1])

# For comparison, let's also run OLS (which might be biased)
X_with_const = sm.add_constant(X)
ols_reg = sm.OLS(Z, X_with_const).fit()

print(f"\nCOMPARISON WITH OLS (potentially biased):")
print("OLS Estimate:", ols_reg.params.iloc[1])  # Use iloc
print("IV Estimate:", second_stage.params.iloc[1])
print("Difference:", abs(ols_reg.params.iloc[1] - second_stage.params.iloc[1]))

# Proper 2SLS using statsmodels
print(f"\nUSING STATSMODELS 2SLS:")
# Note: In practice, you'd use this method
iv_reg = IV2SLS(Z, sm.add_constant(X), sm.add_constant(Y)).fit()
print("2SLS Coefficient:", iv_reg.params.iloc[1])  # Use iloc

Dataset shape: (10000, 4)

Column names: ['Unnamed: 0', 'Z', 'X', 'Y']

First few rows:
   Unnamed: 0  Z  X         Y
0           0  1  0  0.056584
1           1  1  1  1.387519
2           2  0  1  4.289658
3           3  0  1  0.743003
4           4  0  1  2.718527

Basic statistics:
        Unnamed: 0             Z             X             Y
count  10000.00000  10000.000000  10000.000000  10000.000000
mean    4999.50000      0.497600      0.613600      0.687416
std     2886.89568      0.500019      0.486948      2.312184
min        0.00000      0.000000      0.000000    -10.534506
25%     2499.75000      0.000000      0.000000     -0.855894
50%     4999.50000      0.000000      1.000000      0.682283
75%     7499.25000      1.000000      1.000000      2.214232
max     9999.00000      1.000000      1.000000     10.345296

Missing values:
Unnamed: 0    0
Z             0
X             0
Y             0
dtype: int64

INSTRUMENTAL VARIABLES EXPLANATION

The question asks: "Use instrumen

In [5]:
# First stage: regress X on the instrument Y
X_first_stage       = sm.add_constant(data["Y"])
first_stage_model   = sm.OLS(data["X"], X_first_stage).fit()
data["X_hat"]       = first_stage_model.predict(X_first_stage)

# Second stage: regress Z on the predicted X (X_hat)
X_second_stage      = sm.add_constant(data["X_hat"])
second_stage_model  = sm.OLS(data["Z"], X_second_stage).fit()

# Print  summary of the second stage regression
print(second_stage_model.summary())

                            OLS Regression Results                            
Dep. Variable:                      Z   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                    0.9071
Date:                Wed, 13 Aug 2025   Prob (F-statistic):              0.341
Time:                        22:06:39   Log-Likelihood:                -7257.3
No. Observations:               10000   AIC:                         1.452e+04
Df Residuals:                    9998   BIC:                         1.453e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.3614      0.143      2.525      0.0

<font color = 'plum'>
Instrumental variables (IV) estimation using two-stage least squares (2SLS):

### Estimated Effect of X on Z
- **Coefficient of X (via instrument Y):** **0.222**
- **Standard Error:** **0.233**
- **t-value:** **0.952**
- **p-value:** **0.341**

### Interpretation
- The estimated effect of X on Z is **positive**, but **not statistically significant** at conventional levels (e.g., 5% significance level), since the p-value is **0.341**.
- Based on the instrument Y, there is **no strong evidence** of a causal effect of X on Z.


## Question 2  
**10 Points**  
Use two-stage least squares to find the effect of X2 on Y2 in homework_13.2.csv. Which of these is the closest? 

Option A.
-1.5 

Option B.
-2 

Option C.
-1 

Option D.
 -2.5

In [6]:
data_2 = pd.read_csv('https://raw.githubusercontent.com/joshua-vonkorff/2025-summer-mod-6/refs/heads/main/homework_13.2.csv')
data_2 = data_2.drop(columns=['Unnamed: 0'])

In [7]:
import pandas as pd
import statsmodels.api as sm


# First stage: regress X2 on Z2
X2_model = sm.OLS(data_2['X2'], sm.add_constant(data_2['Z2'])).fit()
data_2['X2_hat'] = X2_model.predict(sm.add_constant(data_2['Z2']))

# Second stage: regress Y2 on predicted X2 (X2_hat)
Y2_model = sm.OLS(data_2['Y2'], sm.add_constant(data_2['X2_hat'])).fit()

# Print the summary of the second stage regression
print(Y2_model.summary())



                            OLS Regression Results                            
Dep. Variable:                     Y2   R-squared:                       0.446
Model:                            OLS   Adj. R-squared:                  0.446
Method:                 Least Squares   F-statistic:                     8045.
Date:                Wed, 13 Aug 2025   Prob (F-statistic):               0.00
Time:                        22:06:40   Log-Likelihood:                -24325.
No. Observations:               10000   AIC:                         4.865e+04
Df Residuals:                    9998   BIC:                         4.867e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0015      0.028      0.054      0.9

<font color = 'plum'>

Two-stage least squares (2SLS) regression analysis to estimate the effect of **X2 on Y2**, using **Z2 as an instrument**:

### **Second Stage Regression Summary**
- **Dependent Variable**: Y2
- **Independent Variable**: Predicted X2 (X2_hat from first stage)
- **Sample Size**: 10,000 observations

#### **Key Coefficients**
| Term     | Coefficient | Std. Error | t-value | p-value | 95% CI             |
|----------|-------------|------------|---------|---------|--------------------|
| Intercept| 0.0015      | 0.028      | 0.054   | 0.957   | [-0.053, 0.056]    |
| X2_hat   | **-2.5026** | 0.028      | -89.694 | <0.001  | [-2.557, -2.448]   |

### **Interpretation**
- The coefficient of **X2_hat** is **-2.5026**, which is **statistically significant** (p < 0.001).
-  a one-unit increase in **X2** (as predicted by the instrument **Z2**) is associated with a **2.5 unit decrease in Y2**, on average.
- The high **t-value** and low **p-value** indicate strong evidence *against* the null hypothesis (that the coefficient is zero).


## Question 3  
**10 Points**  

In homework_13.3.csv, which are the compliers? Recall that for a complier, changing the Z value from 0 to 1 would increase the likely value of X. Here,` ZW_int = Z3 * W3`. All confounders are given in this case, so you can use linear regression to understand the situation. (2SLS is not necessary.) 

Option A
The compliers are those with negative numbers in the W3 column.

Option B
The compliers are those with positive numbers in the Z3 column.

Option C
The compliers are those with positive numbers in the W3 column. 

Option D
The compliers are those with negative numbers in the Z3 column. 



In [8]:

data_3 = pd.read_csv('https://raw.githubusercontent.com/joshua-vonkorff/2025-summer-mod-6/refs/heads/main/homework_13.3.csv')
data_3 = data_3.drop(columns=['Unnamed: 0'])

In [9]:


# Fit a linear regression model to predict X3 using Z3, W3, and their interaction
model = smf.ols("X3 ~ Z3 + W3 + Z3:W3", data=data_3).fit()

# Display the regression summary
print(model.summary())

# Interpret the interaction term coefficient
interaction_coef = model.params["Z3:W3"]
if interaction_coef > 0:
    print("Compliers are those with positive values in the W3 column.")
else:
    print("Compliers are those with negative values in the W3 column.")



                            OLS Regression Results                            
Dep. Variable:                     X3   R-squared:                       0.749
Model:                            OLS   Adj. R-squared:                  0.749
Method:                 Least Squares   F-statistic:                     9943.
Date:                Wed, 13 Aug 2025   Prob (F-statistic):               0.00
Time:                        22:06:41   Log-Likelihood:                -14211.
No. Observations:               10000   AIC:                         2.843e+04
Df Residuals:                    9996   BIC:                         2.846e+04
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.0005      0.010     -0.047      0.9

<font color = 'plum'>
Based on the linear regression analysis of **X3** on **Z3**, **W3**, and their interaction (**Z3:W3**), we find the following key coefficients:

- **Z3 coefficient**: +1.0115  
- **W3 coefficient**: +0.9934  
- **Z3:W3 interaction coefficient**: **−0.9969**

This means the effect of **Z3** on **X3** depends on the value of **W3**. Specifically, the **marginal effect** of Z3 on X3 is:

$$
\frac{\partial X3}{\partial Z3} = 1.0115 - 0.9969 \cdot W3
$$

So, increasing Z3 from 0 to 1 **increases X3** when this expression is **positive**, which happens when:

$$
W3 < \frac{1.0115}{0.9969} \approx 1.015
$$

Thus, **compliers** are those for whom **W3 is less than approximately 1.015**, and especially those with **negative W3 values** are most likely to be compliers.

### ✅ Correct answer: **Option A**  
> The compliers are those with negative numbers in the W3 column.


## Question 4  
**10 Points**  

    Z4 = np.random.normal(0, 1, num)
    W4 = np.random.normal(0, 1, num)
    ZW_int = W4 * Z4
    X4 = Z4 + W4 - 2 * ZW_int + np.random.normal(0, 1, num)
    Y4 = W4 + X4 - Z4 + np.random.normal(0, 1, num)

Is the instrument Z4 valid? If not, which of the following is a reason that it is invalid? 

Option A. 
It is invalid because the interaction term ZW_int affects X4.

Option B. 
 It is valid.

Option C. 
It is invalid because Z4 affects X4.

Option D. 
It is invalid because Z4 affects Y4.



In [10]:
num = 10000

Z4 = np.random.normal(0, 1, num)         # Instrument candidate
W4 = np.random.normal(0, 1, num)         # Another variable
ZW_int = W4 * Z4                         # Interaction term
X4 = Z4 + W4 - 2 * ZW_int + np.random.normal(0, 1, num)  # Endogenous regressor
Y4 = W4 + X4 - Z4 + np.random.normal(0, 1, num)          # Outcome variable


In [11]:


# Define the independent variables including the interaction term
X = data_3[["Z3", "ZW_int", "W3"]]
X = sm.add_constant(X)

# Define the dependent variable
y = data_3["X3"]

# Fit the linear regression model
model = sm.OLS(y, X).fit()

# Predict the effect of Z3 on X3 by computing the partial derivative
# For each row, the effect of Z3 is: coef(Z3) + coef(ZW_int) * W3
effect_of_Z3 = model.params["Z3"] + model.params["ZW_int"] * data_3["W3"]

# Identify compliers: those for whom the effect of Z3 on X3 is positive
compliers = data_3[effect_of_Z3 > 0]

# Display the number of compliers and a preview
print(f"Number of compliers: {len(compliers)}")
print(compliers.head())



Number of compliers: 8353
         Z3        X3    ZW_int        W3        Y3
1  0.352335  0.035811 -0.147173 -0.417707  0.110531
2 -0.459113 -0.125720 -0.222680  0.485023 -0.550161
4 -0.770318  0.402082  0.275651 -0.357840  1.360916
5  0.100572 -0.113576  0.024532  0.243924  0.397228
6 -0.975992 -2.376395  0.986451 -1.010716 -2.846744


<font color = 'plum'>To determine whether Z4 is a valid instrument for X4 in the regression of Y4 on X4, we need to assess the two key conditions for instrument validity:

1. Relevance: The instrument must be correlated with the endogenous regressor (here, X4).
2. Exogeneity: The instrument must not be correlated with the error term in the outcome equation (i e., it should not directly affect Y4 except through X4).

- Z4 affects X4 directly: Yes, Z4 is part of the formula for X4. So relevance is satisfied.

- Z4 affects Y4 directly: Yes, Z4 appears directly in the formula for Y4 (specifically, -Z4), which violates the exogeneity condition. This means Z4 has a direct effect on Y4 that is not mediated through X4.


Therefore:
- Z4 is not a valid instrument.
- The correct reason is: Z4 affects Y4 directly, violating the exogeneity condition.